# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiknaxTheGreek/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task

The project is a **content-performance prioritisation POC**.

The primary task is **ranking / scoring**: deciding which content pages should be reviewed first.

Two supervised prediction tasks feed that ranking:

1. **Classification** — estimate whether a page will experience a future decline.
2. **Regression** — estimate the magnitude of the future change.
3. **Ranking / scoring** — combine predicted decline risk and predicted severity to prioritise pages for human review.

Before modeling, interpretable **grouping / stratification** is used as data design rather than as a learned ML task: pages may be balanced across exposure groups, features are organised into families before reduction, and client groups define validation boundaries.

Signal analysis and peer-relative comparisons remain supporting feature-analysis and diagnostic steps.

The locked workflow is:

**population grouping / stratification → feature reduction → classification + regression → ranking / scoring → diagnostic reason codes / action archetypes**

The final output is a ranked human-review queue. Diagnostic archetypes are assigned after ranking from transparent review signals; they are not learned clusters.

In [1]:
import pandas as pd
from pathlib import Path

candidate_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Unique content items: {df['content_id'].nunique():,}")
print(f"Pseudonymized clients: {df['client_id'].nunique():,}")

assert df['content_id'].nunique() == len(df), (
    "Expected one row per pseudonymized content item."
)

pipeline_steps = [
    "population grouping / stratification (data design)",
    "future-decline classification",
    "signed future-change regression",
    "ranking / scoring",
    "diagnostic reason codes / action archetypes",
]
print("Project pipeline:", " -> ".join(pipeline_steps))


Rows: 30,000
Columns: 44
Unique content items: 30,000
Pseudonymized clients: 32
Project pipeline: population grouping / stratification (data design) -> future-decline classification -> signed future-change regression -> ranking / scoring -> diagnostic reason codes / action archetypes


## 2. Target or proxy

### Grouping / stratification

Grouping is a **data-design step, not a target-bearing ML task**.

It can be used to:
- balance the proof-of-concept population across exposure levels;
- organise candidate variables into interpretable feature families;
- define client groups for honest validation;
- support peer-relative diagnostic comparisons.

These groups are defined transparently from pre-decision information. They are not learned clusters.

### Classification

The classification target is an observed future outcome such as:

`future_decline_30d`

where:

- `1` = future decline;
- `0` = no future decline.

Classification answers:

**Will the page decline?**

### Regression

The regression target is the **signed future relative change**, such as:

`future_relative_change_30d`

Regression answers:

**How much is performance expected to change, and in which direction?**

For ranking, a negative regression prediction can then be transformed into a non-negative **predicted decline severity**. Positive predicted change contributes zero decline severity rather than being mislabeled as a decline.

### Ranking / scoring

The final ranking does not use a manually created priority label.

It combines model outputs that answer complementary questions:

**predicted decline risk + severity derived from negative predicted change → review priority**

so that limited human attention is allocated to the pages of greatest apparent concern.

### Time structure

The final targets must use separate past and future periods:

**past features → decision point → future outcome**

The exact windows and target definitions will be fixed in the data-contract stage.

### Starter-data proxy

The starter dataset is only a trailing-90-day snapshot, so it cannot provide a true future target.

For Assignment 3 only:

`decline_proxy = 1` when `trend_direction == "down"`

This is a temporary framing proxy, not a true future label.

Because `trend_direction` is derived from `trend_pct`, neither field may be used as a predictive feature when this proxy is used.

In [2]:
# Transparent starter-data proxy for Assignment 3 framing.
required_proxy_columns = {
    "content_id",
    "trend_direction",
    "trend_pct",
    "impressions_prev_30d",
    "impressions_last_30d",
}
missing = sorted(required_proxy_columns.difference(df.columns))
assert not missing, f"Missing required proxy columns: {missing}"

proxy_frame = df[[
    "content_id",
    "impressions_prev_30d",
    "impressions_last_30d",
    "trend_pct",
    "trend_direction",
]].copy()

proxy_frame["decline_proxy"] = (
    proxy_frame["trend_direction"].str.lower().eq("down").astype("int8")
)

print("Starter proxy: decline_proxy = 1 when trend_direction == 'down'.")
print("This is a current-window proxy, not a future causal or intervention label.\n")
print(proxy_frame["decline_proxy"].value_counts().sort_index())
print(f"Proxy positive rate: {proxy_frame['decline_proxy'].mean():.1%}\n")

display(proxy_frame.head(10))

print("Excluded from predictive features for this proxy: ['trend_direction', 'trend_pct']")

# Later warehouse target structure (not fabricated from this snapshot):
target_schema = pd.DataFrame({
    "field": [
        "future_decline_30d",
        "future_relative_change_30d",
    ],
    "role": [
        "future-state classification target",
        "signed future-relative-change regression target",
    ],
    "available_in_starter_snapshot": [False, False],
})
display(target_schema)


Starter proxy: decline_proxy = 1 when trend_direction == 'down'.
This is a current-window proxy, not a future causal or intervention label.

decline_proxy
0    13738
1    16262
Name: count, dtype: int64
Proxy positive rate: 54.2%



,content_id,impressions_prev_30d,impressions_last_30d,trend_pct,trend_direction,decline_proxy
0,content_304f48230142,987,578,-41.4,down,1
1,content_a1fb4e703a9e,5915,2501,-57.7,down,1
2,content_9aa793d4d895,6089,2382,-60.9,down,1
3,content_331d6c4de07b,4206,3626,-13.8,stable,0
4,content_d99b7a2d90ca,6452,4211,-34.7,down,1
5,content_d4084a4bc775,1009,617,-38.9,down,1
6,content_9a34b442b552,13,1,-92.3,down,1
7,content_a63219c6e95a,632,636,0.6,stable,0
8,content_5e6c160719bc,13828,5696,-58.8,down,1
9,content_c27558df2b0c,356,252,-29.2,down,1


Excluded from predictive features for this proxy: ['trend_direction', 'trend_pct']


,field,role,available_in_starter_snapshot
0,future_decline_30d,future-state classification target,False
1,future_relative_change_30d,signed future-relative-change regression target,False


## 3. Success metrics

The three benchmarked predictive components each have one primary metric and supporting metrics.

| Task | Primary metric | Secondary metrics |
|---|---|---|
| **Classification** | **ROC-AUC** | Precision, Recall, F1 Score |
| **Regression** | **RMSE** | MAE, Median Absolute Error, R² |
| **Ranking / scoring** | **Precision@50** | Recall@50, Lift@50, NDCG@50 |

Grouping / stratification has **no standalone ML metric** because it is a transparent population and validation design step rather than a learned clustering model. Its correctness is checked through balance, coverage, deterministic construction, and zero client overlap where required.

### Classification

**ROC-AUC** is the primary metric because it measures how well the model separates future declines from non-declines without requiring a fixed classification threshold.

Precision, Recall and F1 are reported to show the practical balance between false alarms and missed declines.

### Regression

**RMSE** is the primary metric because large errors in predicted future change should be penalised more strongly.

The model must beat a simple baseline that predicts the training-set mean future change.

MAE, Median Absolute Error and R² provide additional views of prediction error and explained variation.

### Ranking / scoring

**Precision@50** is the primary project metric because the final output is a limited human-review queue.

It measures:

**Of the top 50 recommended pages, how many are genuinely relevant future cases?**

Recall@50 measures coverage, Lift@50 compares the queue with the overall base rate, and NDCG@50 checks whether relevant cases appear near the top.

The learned ranking must be compared with the frozen rule-based baseline using the same pages, future outcomes and `K = 50`.

`K = 50` is the fixed reporting depth for this POC. A real operational review capacity can replace it later if one is provided.

### Supporting analysis

Permutation importance may be used to inspect useful features.

Peer-relative analyses may be used to test whether unusually weak pages within comparable pre-decision groups are enriched for future declines.

These are supporting analyses, not separate ML tasks.

In [3]:
# Metric contract for the three benchmarked predictive components.
import pandas as pd

metric_contract = pd.DataFrame([
    ("Classification", "ROC-AUC", "Precision; Recall; F1 Score"),
    ("Regression", "RMSE", "MAE; Median Absolute Error; R²"),
    ("Ranking / scoring", "Precision@50", "Recall@50; Lift@50; NDCG@50"),
], columns=["task", "primary_metric", "secondary_metrics"])

display(metric_contract)
print("Primary project metric: Precision@50")
print("Grouping / stratification is a transparent data-design step, not a learned clustering task.")
print("Ranking comparison: learned queue vs frozen rule baseline on the same future outcomes.")
print("K = 50 is the fixed reporting depth for this POC.")


,task,primary_metric,secondary_metrics
0,Classification,ROC-AUC,Precision; Recall; F1 Score
1,Regression,RMSE,MAE; Median Absolute Error; R²
2,Ranking / scoring,Precision@50,Recall@50; Lift@50; NDCG@50


Primary project metric: Precision@50
Grouping / stratification is a transparent data-design step, not a learned clustering task.
Ranking comparison: learned queue vs frozen rule baseline on the same future outcomes.
K = 50 is the fixed reporting depth for this POC.


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one pseudonymized content page at a defined decision point**.

Each row should contain only information that would have been available before that decision point, such as:

- search and traffic performance;
- engagement signals;
- content characteristics;
- freshness or update history;
- transparent grouping / stratification context derived from pre-decision data;
- and peer-relative features created from past data.

The starter dataset contains **30,000 rows and 30,000 unique `content_id` values**, so it currently has one row per pseudonymized content page.

For the final warehouse version, the same page may appear at different decision dates. In that case, the unit becomes:

**one content page × one decision point**

The future classification and regression targets belong to the same row but are calculated only from the later outcome window.

The required structure is therefore:

**page at decision point → past features → future decline outcome + future change magnitude**

Identifiers such as `content_id` and `client_id` may be used for joining, grouping and validation splits, but they are not predictive features.

In [4]:
# Verify the starter-data grain and show a compact example of one-row-per-page data.
unit_cols = [
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update",
]

missing_unit_cols = [c for c in unit_cols if c not in df.columns]
assert not missing_unit_cols, f"Missing unit-of-analysis columns: {missing_unit_cols}"

n_rows = len(df)
n_content = df["content_id"].nunique()

print(f"Rows: {n_rows:,}")
print(f"Unique content_id values: {n_content:,}")
print("Starter unit of analysis: one pseudonymized content page per row")

assert n_rows == n_content, "Starter data is not one row per content page."

print("Identifier roles:")
print("  content_id -> join/group key, not a predictive feature")
print("  client_id  -> grouping/split key, not a predictive feature")

display(df[unit_cols].head())


Rows: 30,000
Unique content_id values: 30,000
Starter unit of analysis: one pseudonymized content page per row
Identifier roles:
  content_id -> join/group key, not a predictive feature
  client_id  -> grouping/split key, not a predictive feature


,content_id,client_id,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,engagement_rate,content_age_days,days_since_last_update
0,content_304f48230142,client_f369cb89fc,3803,29,17,0.76,10.6,5.88,187,20
1,content_a1fb4e703a9e,client_4e07408562,15320,7,9,0.05,20.3,0.00,445,25
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,11,0.09,36.5,0.00,141,20
3,content_331d6c4de07b,client_19581e27de,11751,58,78,0.49,6.2,1.28,463,22
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,145,0.13,44.0,0.00,263,14


## 5. Why ML beats a fixed rule here

A fixed rule is useful as a simple baseline, but it is unlikely to capture the full problem.

Content performance depends on several signals at the same time, including visibility, clicks, engagement, freshness, page type and recent performance. The same value can also mean different things for different kinds of pages.

For example, weak CTR may be normal for one type of page but unusual for another. A decline may also matter more when it affects a high-visibility page than a low-visibility page.

The ML approach can combine these signals, learn different page patterns, estimate future decline risk and decline size, and use that information to rank pages.

The fixed rule is still important. It provides a transparent baseline that the ML approach must beat.

If the ML pipeline does not improve the final ranking compared with the simple rule, the simpler rule should be preferred.

In [5]:
# Simple checks supporting the case for a multivariable ML POC.
supporting_cols = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update",
    "content_type",
]

missing_supporting = [c for c in supporting_cols if c not in df.columns]
assert not missing_supporting, f"Missing supporting columns: {missing_supporting}"

print(f"Supporting feature types available: {len(supporting_cols)}")
print(f"Observed content types: {df['content_type'].nunique(dropna=True)}")
print("Baseline rule remains the comparator; ML must improve the final ranking to be kept.")


Supporting feature types available: 8
Observed content types: 3
Baseline rule remains the comparator; ML must improve the final ranking to be kept.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.